# Notebook 04b — Automated Calibration Pair Annotation

**Purpose:** Annotate the 293 calibration pairs generated in `04_esco_calibration.ipynb` using GPT-4o-mini as the judge.

Each pair is `(extracted_skill_phrase, top1_esco_label)`. GPT-4o-mini decides whether the extracted phrase and the ESCO concept refer to the same skill or competence.

**Why automated annotation:**  
Manual annotation of 293 pairs is time-consuming and introduces human inconsistency. LLM-as-annotator is an accepted practice in NLP research when annotation cost is high and the task is well-defined. The annotation prompt is kept simple and deterministic (temperature=0).

**Input:** `data/processed/esco/calibration_pairs.csv`  
**Output:** `data/processed/esco/calibration_pairs.csv` (with `is_match` column filled)

**Cost estimate:** ~293 calls × ~50 tokens = ~15,000 tokens → under $0.01 with GPT-4o-mini.

In [1]:
import os
from pathlib import Path
# Set working directory to project root regardless of launch location
_nb_path = globals().get("__vsc_ipynb_file__") or globals().get("__file__", None)
if _nb_path:
    # Launched from VS Code or as script — go up from notebooks/3_analysis/
    os.chdir(Path(_nb_path).resolve().parent.parent.parent)
elif not (Path.cwd() / "data").exists():
    # Launched from wrong cwd — try to find project root
    _root = Path.cwd()
    for _ in range(4):
        if (_root / "data").exists():
            break
        _root = _root.parent
    os.chdir(_root)
assert (Path.cwd() / "data").exists(), f"Cannot find project root from {Path.cwd()}"

import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

CAL_PAIRS = Path('data/processed/esco/calibration_pairs.csv')
df = pd.read_csv(CAL_PAIRS)

print(f'Loaded {len(df)} calibration pairs')
print(f'Already annotated: {df["is_match"].notna().sum()}')
print(f'Remaining: {df["is_match"].isna().sum()}')
print()
print('Band distribution:')
print(df['sim_band'].value_counts().sort_index())

Loaded 293 calibration pairs
Already annotated: 293
Remaining: 0

Band distribution:
sim_band
0.60-0.65    43
0.65-0.70    43
0.70-0.75    43
0.75-0.80    43
0.80-0.85    35
<0.60        43
>0.85        43
Name: count, dtype: int64


## Annotation function

Each pair is sent to GPT-4o-mini with a strict prompt. Temperature is set to 0 for deterministic output. The model returns only `1` (match) or `0` (no match).

In [2]:
SYSTEM_PROMPT = """You are an expert in occupational skills and competences.
Your task is to judge whether a short extracted skill phrase refers to the same skill or competence as a given ESCO concept label.

Rules:
- Answer only with 1 (yes, same skill) or 0 (no, different skill).
- A match means the phrase and the ESCO label refer to the same underlying skill or competence, even if worded differently.
- Partial overlaps, broad/narrow mismatches, or unrelated concepts are NOT matches.
- Do not explain. Output only 1 or 0."""

def annotate_pair(extracted_skill: str, esco_label: str) -> int:
    """Ask GPT-4o-mini if extracted_skill matches esco_label. Returns 1 or 0."""
    user_msg = f'Extracted phrase: "{extracted_skill}"\nESCO concept: "{esco_label}"\nIs this a match? (1 or 0)'
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg},
        ],
        temperature=0,
        max_tokens=1,
    )
    answer = response.choices[0].message.content.strip()
    return 1 if answer == '1' else 0

# Quick smoke test
test = annotate_pair('machine learning', 'machine learning')
print(f'Smoke test (machine learning vs machine learning): {test}  (expected 1)')
test2 = annotate_pair('bicycle repair', 'machine learning')
print(f'Smoke test (bicycle repair vs machine learning): {test2}  (expected 0)')

Smoke test (machine learning vs machine learning): 1  (expected 1)
Smoke test (bicycle repair vs machine learning): 0  (expected 0)


## Run annotation

Annotates all unannotated pairs. Saves after every 25 pairs so progress is not lost on interruption.

In [3]:
SAVE_EVERY = 25
DELAY = 0.1  # seconds between calls to avoid rate limits

pending = df[df['is_match'].isna()].index.tolist()
print(f'Annotating {len(pending)} pairs...')

for i, idx in enumerate(pending):
    row = df.loc[idx]
    result = annotate_pair(str(row['extracted_skill']), str(row['top1_esco_label']))
    df.at[idx, 'is_match'] = result
    df.at[idx, 'annotator_notes'] = 'gpt-4o-mini'

    if (i + 1) % SAVE_EVERY == 0:
        df.to_csv(CAL_PAIRS, index=False)
        print(f'  [{i+1}/{len(pending)}] saved checkpoint')

    time.sleep(DELAY)

df.to_csv(CAL_PAIRS, index=False)
print(f'\nDone. All {len(df)} pairs annotated and saved.')

Annotating 0 pairs...

Done. All 293 pairs annotated and saved.


## Review annotation results

In [4]:
df['is_match'] = df['is_match'].astype(int)

print('=== Overall ===')
print(f"Total pairs:   {len(df)}")
print(f"Matches:       {df['is_match'].sum()} ({df['is_match'].mean()*100:.1f}%)")
print(f"Non-matches:   {(df['is_match']==0).sum()}")
print()

print('=== Match rate by similarity band ===')
band_order = ['<0.60','0.60-0.65','0.65-0.70','0.70-0.75','0.75-0.80','0.80-0.85','>0.85']
for band in band_order:
    subset = df[df['sim_band'] == band]
    if len(subset) == 0:
        continue
    match_rate = subset['is_match'].mean() * 100
    print(f"  {band:12s}: {match_rate:5.1f}% match  (n={len(subset)})")

print()
print('=== Sample matches (high similarity) ===')
print(df[df['is_match']==1].sort_values('cosine_similarity', ascending=False)
      [['extracted_skill','top1_esco_label','cosine_similarity']].head(8).to_string(index=False))

print()
print('=== Sample non-matches (high similarity — false positives) ===')
print(df[(df['is_match']==0) & (df['cosine_similarity']>0.70)]
      [['extracted_skill','top1_esco_label','cosine_similarity']].head(8).to_string(index=False))

=== Overall ===
Total pairs:   293
Matches:       121 (41.3%)
Non-matches:   172

=== Match rate by similarity band ===
  <0.60       :   9.3% match  (n=43)
  0.60-0.65   :  11.6% match  (n=43)
  0.65-0.70   :  25.6% match  (n=43)
  0.70-0.75   :  34.9% match  (n=43)
  0.75-0.80   :  48.8% match  (n=43)
  0.80-0.85   :  74.3% match  (n=35)
  >0.85       :  90.7% match  (n=43)

=== Sample matches (high similarity) ===
   extracted_skill    top1_esco_label  cosine_similarity
alternative energy alternative energy                1.0
    cryptocurrency     cryptocurrency                1.0
              perl               Perl                1.0
           chinese            Chinese                1.0
           biology            biology                1.0
            energy             energy                1.0
         ukrainian          Ukrainian                1.0
          kinetics           kinetics                1.0

=== Sample non-matches (high similarity — false positives) ===
  

## Manual validation of GPT annotations

A stratified sample of 35 pairs (5 per similarity band) was reviewed manually after automated annotation.

**Result: 33/35 agreement (94.3% accuracy)**

Two corrections were applied:

| Extracted phrase | ESCO label | GPT | Human | Reason |
|---|---|---|---|---|
| `connects ecommerce erp` | `e-commerce systems` | 1 | 0 | Phrase describes ERP-ecommerce integration, not the e-commerce systems skill |
| `chemical data analysis` | `analyse chemical substances` | 1 | 0 | Analysing chemical *data* ≠ analysing chemical *substances* (lab skill) |

These corrections were applied directly to `calibration_pairs.csv` (`annotator_notes` = `gpt-4o-mini; corrected by human reviewer`).

The 94.3% agreement rate validates the automated annotation approach as reliable for threshold calibration purposes.

In [5]:
# Apply manual corrections from human review of 35-pair stratified sample
df = pd.read_csv(CAL_PAIRS)
df['is_match'] = df['is_match'].astype(int)

corrections = [
    ('connects ecommerce erp', 'e-commerce systems'),
    ('chemical data analysis', 'analyse chemical substances'),
]

for skill, esco in corrections:
    mask = (df['extracted_skill'] == skill) & (df['top1_esco_label'] == esco)
    df.loc[mask, 'is_match'] = 0
    df.loc[mask, 'annotator_notes'] = 'gpt-4o-mini; corrected by human reviewer'
    print(f'Corrected: "{skill}" → "{esco}"')

df.to_csv(CAL_PAIRS, index=False)
print(f'\nFinal: {df["is_match"].sum()} matches / {len(df)} total ({df["is_match"].mean()*100:.1f}%)')

Corrected: "connects ecommerce erp" → "e-commerce systems"
Corrected: "chemical data analysis" → "analyse chemical substances"

Final: 121 matches / 293 total (41.3%)


## Next step

Annotation is complete. Return to `04_esco_calibration.ipynb` and run cells **17–19** (the threshold sweep).

Those cells will read this annotated file and compute Precision / Recall / F1 across thresholds 0.60–0.85 to find the optimal cutoff.